In [1]:
from gomoku import Gomoku
from mcts import Tree
import numpy as np

In [2]:
class Player:
    def next_move(self, game: Gomoku) -> tuple[int, int]:
        raise NotImplementedError

class RandomPlayer(Player):
    def next_move(self, game: Gomoku) -> tuple[int, int]:
        moves = game.actions()
        random_move  = np.random.randint(0, len(moves))
        return moves[random_move]
    
class FlatPlayer(Player):
    def __init__(self, max_moves: int = 0):
        self.max_moves = max_moves
    
    def next_move(self, game: Gomoku):
        moves = game.actions()
        if self.max_moves:
            moves = moves[:self.max_moves]
        
        max_reward = -np.inf
        argmax_reward = None
        
        for move in moves:
            new_game = game.copy()
            reward, _ = new_game.play(move)
            if reward > max_reward:
                max_reward = reward
                argmax_reward = move
        
        if argmax_reward is not None:
            move = argmax_reward
        else:
            move = moves[np.random.randint(0, len(moves))]
        
        return move
            
class MCTSPlayer(Player):
    def __init__(self, iterations=1000):
        self.iterations = iterations

    def next_move(self, game: Gomoku):
        tree = Tree(game)
        for _ in range(self.iterations):
            node = tree.select()
            value = tree.simulate(node)
            tree.backpropagate(node, value)
        best_child = max(tree.root.children, key=lambda child: child.Q)
        return best_child.state.history[-1]

In [12]:
game = Gomoku(
    M=5,
    N=5,
    K=3,
    FIRST_PLAYER=1,
)

player1 = FlatPlayer()
player2 = MCTSPlayer()
while not game.fin():
    if game.player == 1:
        move = player1.next_move(game)
    else:
        move = player2.next_move(game)
    game.play(move)
    
game.print()

Winner: -1
. X . . .
. X . . .
. O O O .
. . . . .
X . . . .


In [13]:
game.history

[(1, 1), (2, 2), (4, 0), (2, 1), (0, 1), (2, 3)]